# Inference Parameters and Text Generation

After training, a language model does not emit finished sentences directly. At each step it outputs **logits** — raw scores over the entire vocabulary — and your application must **decode** those scores into the next token. The choice of decoding strategy and inference parameters determines whether output is precise or creative, concise or verbose, diverse or repetitive.

These parameters — **temperature**, **top-k**, **top-p**, **max tokens**, **stop sequences**, **frequency penalty**, **presence penalty**, and **seed** — are applied at **inference time**. They do not change model weights; they shape the probability distribution from which each token is chosen. The same model with different settings can produce a rigid factual answer or a flowing creative draft.

This notebook explains each parameter in depth: what it does mathematically, how it affects output, typical value ranges, when to use it, and how parameters interact. Interactive NumPy examples let you see distributions change as you adjust each knob.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

# Shared toy vocabulary and logits used across examples
vocab = np.array(["the", "cat", "sat", "on", "mat", "dog", "ran"])
base_logits = np.array([2.5, 2.0, 1.6, 1.2, 1.1, 1.0, 0.7])

def softmax(logits):
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()

def probs_with_temperature(logits, temperature):
    t = max(float(temperature), 1e-6)
    return softmax(logits / t)

def sample_top_k(probs, k, rng):
    idx = np.argsort(probs)[::-1][:k]
    p = probs[idx] / probs[idx].sum()
    return rng.choice(idx, p=p)

def sample_top_p(probs, p_threshold, rng):
    order = np.argsort(probs)[::-1]
    sorted_probs = probs[order]
    cdf = np.cumsum(sorted_probs)
    cutoff = int(np.searchsorted(cdf, p_threshold)) + 1
    keep = order[:cutoff]
    renorm = probs[keep] / probs[keep].sum()
    return rng.choice(keep, p=renorm)

def apply_penalties(logits, token_counts, alpha_freq=0.0, alpha_pres=0.0):
    adjusted = logits.copy().astype(float)
    for i, c in enumerate(token_counts):
        adjusted[i] -= alpha_freq * c
        adjusted[i] -= alpha_pres * (1 if c > 0 else 0)
    return adjusted

---

## 1. How Text Generation Works at Inference

Autoregressive language models generate text one **token** at a time. The process repeats until a stopping condition is met:

```
Prompt tokens  →  Model  →  Logits (scores per vocabulary item)
                              ↓
                         Softmax + decoding rules
                              ↓
                         Next token chosen
                              ↓
                    Append to context, repeat
```

At each step the model outputs a vector of **logits** $z = (z_1, z_2, \ldots, z_V)$ — one raw score per vocabulary token. Logits are converted to probabilities with the **softmax** function:

$$P_i = \frac{\exp(z_i)}{\sum_{j=1}^{V} \exp(z_j)}$$

The next token is then selected according to a **decoding strategy**:

- **Greedy** — always pick $\arg\max_i P_i$ (highest probability)
- **Sampling** — draw a token randomly, weighted by $P_i$
- **Beam search** — keep several candidate sequences and score them jointly

Inference parameters (temperature, top-k, top-p, penalties) modify $z$ or $P$ **before** the final selection. They are the primary levers you control when calling APIs such as OpenAI, Anthropic, or local inference servers.

---

## 2. The Full Decoding Pipeline

Most production systems apply parameters in a fixed order at each generation step:

1. Start with model logits $z$
2. Subtract **frequency** and **presence** penalties based on tokens already generated
3. Divide by **temperature** $T$
4. Apply **softmax** to get probabilities $P$
5. **Truncate** with top-k and/or top-p
6. **Sample** (or take argmax for greedy)
7. Check **stop sequences** and **max tokens**; stop or continue

A compact formula capturing steps 2–4:

$$\tilde{z}_i = \frac{z_i - \alpha_f \cdot c_i - \alpha_p \cdot \mathbf{1}(c_i > 0)}{T}$$

$$P_i = \frac{\exp(\tilde{z}_i)}{\sum_j \exp(\tilde{z}_j)}$$

where $c_i$ is how many times token $i$ has already appeared, $\alpha_f$ is the frequency penalty, $\alpha_p$ is the presence penalty, and $T$ is temperature. Steps 5–7 are applied to $P$ before drawing the next token.

Understanding this pipeline helps you reason about **interactions**: a high temperature and a loose top-p together can produce much more random output than either alone.

---

## 3. Temperature

**What it is:** Temperature $T$ is a positive scalar that divides logits before softmax. It is the most widely used control for balancing **determinism** versus **creativity**.

$$P_i(T) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

**How it works:**

| Temperature | Effect on distribution | Output behavior |
|-------------|------------------------|-----------------|
| $T \to 0^+$ | Sharpest — nearly all mass on one token | Deterministic, repetitive |
| $T = 1$ | Unmodified model distribution | Default model behavior |
| $T > 1$ | Flatter — low-probability tokens gain mass | More diverse, riskier |
| $T \gg 1$ | Nearly uniform | Random, often incoherent |

**Typical ranges:**

- **0.0–0.3** — extraction, classification-style answers, code completion where correctness matters
- **0.5–0.8** — balanced assistants, Q&A with grounding
- **0.9–1.2** — general conversation, brainstorming
- **1.3+** — creative writing, ideation (higher hallucination risk)

**API note:** Some APIs treat $T = 0$ as greedy decoding (argmax) rather than literal division by zero.

**Cautions:** Low temperature does not guarantee factual accuracy — it only narrows the distribution. High temperature increases the chance of selecting low-probability tokens, which may be wrong or off-topic.

In [ ]:
# Temperature: how the probability distribution changes
temps = [0.2, 0.7, 1.0, 1.5]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)
for ax, t in zip(axes.flat, temps):
    p = probs_with_temperature(base_logits, t)
    bars = ax.bar(vocab, p, color="steelblue")
    ax.bar(vocab[0], p[0], color="tab:red")  # highlight top token
    ax.set_title(f"temperature = {t}")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("Probability")
plt.suptitle("Lower T peaks on top token; higher T spreads probability mass", y=1.02)
plt.tight_layout()
plt.show()

# Entropy increases with temperature
def entropy(p):
    p = p[p > 0]
    return -np.sum(p * np.log(p))

t_range = np.linspace(0.1, 2.0, 40)
entropies = [entropy(probs_with_temperature(base_logits, t)) for t in t_range]
plt.figure(figsize=(8, 4))
plt.plot(t_range, entropies, linewidth=2)
plt.xlabel("Temperature")
plt.ylabel("Entropy (nats)")
plt.title("Temperature vs output entropy")
plt.show()

---

## 4. Greedy Decoding

**What it is:** At every step, select the token with the highest probability: $\hat{t} = \arg\max_i P_i$. No randomness is involved.

**When to use:**

- Structured extraction (JSON, labels, parsing)
- Code completion where the most likely continuation is usually correct
- Any task where reproducibility and consistency matter more than variety

**Strengths:** Fast, deterministic, easy to debug. Same prompt always yields the same output (with fixed model version).

**Weaknesses:**

- **Repetition loops** — "The the the..." or cycling phrases
- **Locally optimal, globally suboptimal** — the best next token may not lead to the best full sentence
- **Bland open-ended text** — creative tasks feel flat and predictable

**API mapping:** `temperature=0` typically enables greedy (or near-greedy) behavior in OpenAI-compatible APIs.

In [ ]:
# Greedy always picks the same token; sampling varies
probs = probs_with_temperature(base_logits, temperature=1.0)
greedy_token = vocab[np.argmax(probs)]

rng = np.random.default_rng(0)
sampled = [vocab[rng.choice(len(vocab), p=probs)] for _ in range(20)]

print(f"Greedy pick (always):     {greedy_token!r}")
print(f"Sampled picks (20 runs):  {sampled}")
print(f"Greedy probability:       {probs.max():.4f}")

---

## 5. Random Sampling

**What it is:** Draw the next token from the probability distribution $P$ — token $i$ is selected with probability $P_i$. Combined with temperature, top-k, and top-p, sampling produces varied but controlled output.

**When to use:** Chat assistants, creative writing, brainstorming, marketing copy — any task where natural variation improves user experience.

**Relationship to temperature:** Sampling from a sharp distribution ($T < 1$) behaves almost like greedy; sampling from a flat distribution ($T > 1$) explores unlikely tokens.

**Without truncation:** Raw sampling over 50,000+ vocabulary tokens occasionally picks very low-probability tokens, causing incoherent words. **Top-k** and **top-p** exist to prevent this.

---

## 6. Top-k Sampling

**What it is:** Keep only the $k$ tokens with highest probability, set all others to zero, renormalize, then sample.

**Parameter:** `top_k` — integer, typically 1–100. Setting `top_k=1` is equivalent to greedy decoding.

**How it works:**

1. Sort tokens by probability descending
2. Keep the top $k$
3. Renormalize so the kept probabilities sum to 1
4. Sample from this truncated distribution

**Typical values:**

| top_k | Behavior |
|-------|----------|
| 1 | Greedy — only the single best token |
| 10–40 | Conservative — common in code and factual tasks |
| 50–100 | More diverse — creative tasks |

**Strengths:** Simple, fast, caps the candidate set regardless of distribution shape.

**Weaknesses:** Fixed $k$ does not adapt. When the model is very confident (one token at 99%), top-k=50 still keeps 49 unlikely tokens. When uncertain (flat distribution), top-k=10 may cut off valid options.

**API note:** OpenAI chat models do not expose top-k directly; many open-source servers (vLLM, llama.cpp) do.

In [ ]:
rng = np.random.default_rng(0)
p = probs_with_temperature(base_logits, 1.0)

print("Top-k sampling — 12 draws per setting:\n")
for k in [1, 3, 5, 7]:
    picks = [vocab[sample_top_k(p, k, rng)] for _ in range(12)]
    print(f"  top_k={k}: {picks}")

---

## 7. Top-p (Nucleus) Sampling

**What it is:** Also called **nucleus sampling** (Holtzman et al., 2019). Keep the **smallest** set of tokens whose cumulative probability exceeds threshold $p$, then renormalize and sample.

**Parameter:** `top_p` — float in $(0, 1]$, often 0.85–0.95.

**How it works:**

1. Sort tokens by probability descending: $P_{(1)} \geq P_{(2)} \geq \cdots$
2. Find the smallest $n$ such that $\sum_{i=1}^{n} P_{(i)} \geq p$
3. Keep only those $n$ tokens, renormalize, sample

**Why it adapts:** When the model is confident, the top token alone may already exceed $p=0.9$ — nucleus size is 1 (near-greedy). When uncertain, many tokens are needed to reach 0.9 — nucleus expands automatically.

**Typical values:**

| top_p | Behavior |
|-------|----------|
| 0.1–0.5 | Very focused — factual, low diversity |
| 0.85–0.95 | Default range for chat assistants |
| 1.0 | No truncation — full vocabulary eligible |

**Strengths:** Adapts to model confidence per step. Generally preferred over fixed top-k for open-ended generation.

**Cautions:** Very low top_p with low temperature can make output robotic. top_p=1.0 with high temperature allows rare tokens and increases incoherence risk.

In [ ]:
rng = np.random.default_rng(1)
p = probs_with_temperature(base_logits, 1.0)

print("Top-p (nucleus) sampling — 12 draws per setting:\n")
for pth in [0.5, 0.8, 0.95, 1.0]:
    picks = [vocab[sample_top_p(p, pth, rng)] for _ in range(12)]
    # show nucleus size for this distribution
    order = np.argsort(p)[::-1]
    cdf = np.cumsum(p[order])
    nucleus_size = int(np.searchsorted(cdf, pth)) + 1
    print(f"  top_p={pth}: nucleus_size={nucleus_size}, picks={picks}")

---

## 8. Combining Temperature, Top-k, and Top-p

Parameters are usually applied together. A common production starting point:

```python
temperature = 0.7
top_p = 0.9
# top_k often omitted when top_p is set
```

**Interaction rules:**

- **Low T + low top_p** — highly deterministic, best for factual Q&A
- **High T + high top_p** — maximum diversity, highest risk of errors
- **top_k and top_p together** — some runtimes apply both; the stricter constraint wins. Check your inference server documentation.

| Task | temperature | top_p | Notes |
|------|-------------|-------|-------|
| Factual Q&A | 0.0–0.3 | 0.8–0.95 | Prioritize correctness |
| Code generation | 0.0–0.2 | 0.95 | Greedy or near-greedy |
| Chat assistant | 0.7–0.9 | 0.9–0.95 | Balanced default |
| Creative writing | 1.0–1.3 | 0.95–1.0 | More exploration |
| Brainstorming | 1.2–1.5 | 0.95–1.0 | High diversity |

In [ ]:
# Compare decoding presets on the same logits
presets = {
    "Factual (T=0.2, top_p=0.85)": (0.2, 0.85),
    "Balanced (T=0.7, top_p=0.90)": (0.7, 0.90),
    "Creative (T=1.3, top_p=0.95)": (1.3, 0.95),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (name, (t, pth)) in zip(axes, presets.items()):
    p = probs_with_temperature(base_logits, t)
    order = np.argsort(p)[::-1]
    cdf = np.cumsum(p[order])
    n = int(np.searchsorted(cdf, pth)) + 1
    keep = order[:n]
    mask = np.zeros_like(p)
    mask[keep] = p[keep]
    ax.bar(vocab, mask / mask.sum() if mask.sum() else mask, color="steelblue")
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Effective sampling distribution after temperature + top-p", y=1.02)
plt.tight_layout()
plt.show()

---

## 9. Max Tokens

**What it is:** A hard limit on how many tokens the model may **generate** in one response (not including prompt tokens). Called `max_tokens` or `max_completion_tokens` in different APIs.

**Why it matters:**

- **Cost** — API billing is per token; unbounded generation is expensive
- **Latency** — each token requires a forward pass; long outputs are slow
- **Safety** — prevents runaway generation if stop conditions fail
- **UX** — users expect concise answers in many interfaces

**How to choose:**

| Use case | Typical max_tokens |
|----------|-------------------|
| Short answer / classification | 50–256 |
| Chat reply | 512–1024 |
| Summarization | 500–1500 |
| Long-form draft | 2000–4096 |

**Context window interaction:** Total usage is $T_{prompt} + T_{output} \leq T_{window}$. If max_tokens is too large relative to remaining context, the request may fail or truncate input.

**Cautions:** Setting max_tokens too low cuts off mid-sentence. Setting it too high wastes budget when the model would have stopped naturally.

---

## 10. Stop Sequences

**What it is:** A list of strings that, when generated, cause decoding to **stop immediately**. The stop string itself may or may not be included in the returned output depending on the API.

**Common uses:**

- `\n\n` — stop at paragraph boundary
- `"Human:"` / `"User:"` — prevent role-play bleed in chat templates
- `"```"` — end code block generation
- Custom delimiters for structured pipelines

**Example:**

```python
stop=["\n\n", "---", "END"]
```

**Strengths:** Precise control over output length and format without guessing max_tokens.

**Cautions:** Stop strings must match **exact tokenization**. A stop of `"\n\n"` may not fire if the model emits a different whitespace pattern. Test with your target model and tokenizer.

In [ ]:
# Simulated generation with max_tokens and stop sequences
def generate_with_limits(token_stream, max_tokens=10, stop_sequences=None):
    stop_sequences = stop_sequences or []
    output = []
    text = ""
    for i, token in enumerate(token_stream):
        if i >= max_tokens:
            output.append("<MAX_TOKENS>")
            break
        output.append(token)
        text += token + " "
        for stop in stop_sequences:
            if stop in text:
                output.append(f"<STOP:'{stop}'>")
                return output
    return output

stream = ["The", "cat", "sat", "on", "the", "mat", ".", "\n\n", "Then", "it", "ran"]

print("No limits:     ", generate_with_limits(stream, max_tokens=20))
print("max_tokens=5:  ", generate_with_limits(stream, max_tokens=5))
print("stop='\\n\\n':   ", generate_with_limits(stream, max_tokens=20, stop_sequences=["\n\n"]))

---

## 11. Frequency Penalty

**What it is:** Reduces logits of tokens in proportion to how many times they have **already appeared** in the generated text. Penalizes repetition more strongly the more a token is reused.

**Parameter:** `frequency_penalty` — float, typically 0.0–2.0. Default 0.0 (no penalty).

**Effect on logits:**

$$z_i' = z_i - \alpha_f \cdot c_i$$

where $c_i$ is the count of token $i$ in the output so far and $\alpha_f$ is the frequency penalty.

**When to use:** Long-form generation where the model loops phrases — summaries, essays, dialogue. A value of 0.3–0.8 often reduces obvious repetition without distorting meaning.

**Example:** If "the" appeared 5 times, each occurrence adds $0.15$ to the penalty at $\alpha_f = 0.15$, making further "the" tokens less likely.

**Cautions:** Too high a penalty forces the model to avoid common words entirely, producing awkward synonyms or broken grammar.

---

## 12. Presence Penalty

**What it is:** Applies a **fixed** penalty to any token that has appeared at least once, regardless of how many times. Encourages the model to introduce **new** tokens rather than reuse existing ones.

**Parameter:** `presence_penalty` — float, typically 0.0–2.0. Default 0.0.

**Effect on logits:**

$$z_i' = z_i - \alpha_p \cdot \mathbf{1}(c_i > 0)$$

where $\mathbf{1}(c_i > 0)$ is 1 if token $i$ appeared before, else 0.

**Frequency vs presence:**

| Penalty | Scales with | Best for |
|---------|-------------|----------|
| **Presence** | Once per unique token | Encouraging topic diversity |
| **Frequency** | Each repetition | Stopping loops of the same word |

**When to use:** Brainstorming lists, generating varied examples, reducing template-like repetition. Often combined with a small frequency penalty.

**Cautions:** High presence penalty can prevent necessary repetition (e.g., pronouns, technical terms that must recur).

In [ ]:
# Frequency vs presence penalty on logits
counts = np.array([5, 1, 0, 2, 0, 0, 1])  # 'the' appeared 5 times, etc.

scenarios = [
    ("No penalty", 0.0, 0.0),
    ("Frequency=0.15", 0.15, 0.0),
    ("Presence=0.25", 0.0, 0.25),
    ("Both", 0.15, 0.25),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)
for ax, (title, af, ap) in zip(axes.flat, scenarios):
    adj = apply_penalties(base_logits, counts, alpha_freq=af, alpha_pres=ap)
    p = softmax(adj)
    ax.bar(vocab, p, color="steelblue")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Penalties shift probability away from already-used tokens", y=1.02)
plt.tight_layout()
plt.show()

print("Token counts in generated text so far:", dict(zip(vocab, counts)))

---

## 13. Seed (Reproducibility)

**What it is:** An integer seed for the random number generator used in sampling. The same seed + same prompt + same parameters + same model version should produce the same output.

**Parameter:** `seed` — integer. Not all APIs expose this; OpenAI supports it on some models.

**When to use:**

- Debugging — reproduce a bad output exactly
- Regression testing — compare outputs across code changes
- Demos — consistent behavior for presentations

**Limits of reproducibility:**

- Different model versions tokenize or score differently
- GPU floating-point non-determinism in some frameworks
- Batched inference may change ordering
- API-side changes break replay even with fixed seed

**Best practice:** Log model ID, prompt hash, all decoding parameters, and seed with every request for traceability.

In [ ]:
def draw_sequence(seed, temperature=0.9):
    rng = np.random.default_rng(seed)
    probs = probs_with_temperature(base_logits, temperature)
    return [vocab[rng.choice(len(vocab), p=probs)] for _ in range(8)]

print("Same seed -> same sequence:")
print("  seed=42:", draw_sequence(42))
print("  seed=42:", draw_sequence(42))
print("Different seed -> different sequence:")
print("  seed=99:", draw_sequence(99))

---

## 14. Beam Search

**What it is:** Maintains $B$ **beams** — partial sequences — at each step. Expands each beam with all possible next tokens, keeps the $B$ sequences with highest **cumulative log-probability**, and repeats.

**Parameter:** `beam_width` or `num_beams` — typically 3–10.

**When to use:** Machine translation, summarization, captioning — tasks where the best **full sequence** score matters more than diversity.

**Comparison:**

| Method | Picks | Diversity | Cost | Best for |
|--------|-------|-----------|------|----------|
| Greedy | Best next token | Low | Low | Extraction, code |
| Beam | Best $B$ sequences | Low–medium | High | Translation, summarization |
| Sampling | Random from $P$ | High | Low | Chat, creative writing |

**Weaknesses in chat:** Beam search favors high-probability safe phrases, producing generic responses. Rarely used for conversational assistants.

In [ ]:
# Tiny beam search demonstration
beam_vocab = ["A", "B", "C", "<END>"]
idx = {t: i for i, t in enumerate(beam_vocab)}
P = np.array([
    [0.05, 0.55, 0.35, 0.05],
    [0.45, 0.05, 0.35, 0.15],
    [0.35, 0.35, 0.05, 0.25],
    [0.00, 0.00, 0.00, 1.00],
])

def greedy_decode(start="A", steps=5):
    seq, cur = [start], start
    for _ in range(steps):
        cur = beam_vocab[int(np.argmax(P[idx[cur]]))]
        seq.append(cur)
        if cur == "<END>":
            break
    return seq

def beam_search(start="A", steps=5, beam_width=3):
    beams = [([start], 0.0)]
    for _ in range(steps):
        new_beams = []
        for seq, lp in beams:
            cur = seq[-1]
            if cur == "<END>":
                new_beams.append((seq, lp))
                continue
            for j, pj in enumerate(P[idx[cur]]):
                if pj > 0:
                    new_beams.append((seq + [beam_vocab[j]], lp + np.log(pj)))
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
    return beams

print("Greedy:       ", greedy_decode())
print("Beam (B=3):")
for seq, lp in beam_search():
    print(f"  {seq}  log_prob={lp:.4f}")

---

## 15. API Parameter Reference (OpenAI-Compatible)

Mapping of concepts to common API fields:

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `temperature` | float | 1.0 | Randomness: 0 = focused, 2 = very random |
| `top_p` | float | 1.0 | Nucleus sampling threshold |
| `max_tokens` | int | model max | Maximum tokens to generate |
| `stop` | list[str] | null | Strings that halt generation |
| `frequency_penalty` | float | 0.0 | Penalize by repetition count |
| `presence_penalty` | float | 0.0 | Penalize any repeated token |
| `seed` | int | null | RNG seed for reproducibility |
| `n` | int | 1 | Number of completions to return |
| `logprobs` | bool | false | Return log probabilities per token |

**OpenAI recommendation:** Alter either `temperature` or `top_p`, not both aggressively at the same time. Many teams fix `top_p=1.0` and tune only temperature, or fix `temperature=1.0` and tune only top_p.

---

## 16. Full Generation Loop

The following toy implementation combines temperature, top-k, penalties, max tokens, and stop logic in one autoregressive loop. A real LLM replaces the transition matrix with a neural network forward pass.

In [ ]:
toy_vocab = np.array(["I", "like", "to", "learn", "LLMs", ".", "<END>"])
tid = {t: i for i, t in enumerate(toy_vocab)}
T = np.array([
    [0.02, 0.50, 0.25, 0.08, 0.10, 0.03, 0.02],
    [0.02, 0.05, 0.40, 0.20, 0.25, 0.05, 0.03],
    [0.03, 0.05, 0.04, 0.45, 0.35, 0.05, 0.03],
    [0.03, 0.06, 0.08, 0.05, 0.50, 0.20, 0.08],
    [0.02, 0.04, 0.05, 0.12, 0.04, 0.58, 0.15],
    [0.25, 0.22, 0.12, 0.10, 0.08, 0.03, 0.20],
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00],
])

def generate(
    prompt_token="I",
    max_tokens=15,
    temperature=1.0,
    top_k=4,
    alpha_freq=0.10,
    alpha_pres=0.05,
    stop_tokens=("<END>", "."),
    seed=3,
):
    rng = np.random.default_rng(seed)
    out = [prompt_token]
    counts = Counter([prompt_token])
    current = prompt_token

    for _ in range(max_tokens):
        probs = T[tid[current]]
        logits = np.log(np.maximum(probs, 1e-12))
        token_counts = np.array([counts[t] for t in toy_vocab])
        logits = apply_penalties(logits, token_counts, alpha_freq, alpha_pres)
        step_probs = probs_with_temperature(logits, temperature)
        nxt_id = sample_top_k(step_probs, top_k, rng)
        nxt = toy_vocab[nxt_id]
        out.append(nxt)
        counts[nxt] += 1
        current = nxt
        if nxt in stop_tokens:
            break
    return " ".join(out)

print("Default:              ", generate())
print("Low temperature (0.3):  ", generate(temperature=0.3, seed=3))
print("High freq penalty:    ", generate(alpha_freq=0.5, seed=3))
print("Different seed:       ", generate(seed=99))

---

## 17. Tuning by Task Type

| Task | temperature | top_p | max_tokens | frequency_penalty | presence_penalty |
|------|-------------|-------|------------|-------------------|------------------|
| JSON extraction | 0.0 | 0.9 | 500 | 0.0 | 0.0 |
| Code completion | 0.0–0.2 | 0.95 | 1000 | 0.0–0.2 | 0.0 |
| Factual Q&A | 0.0–0.3 | 0.85–0.95 | 512 | 0.0 | 0.0 |
| Chat assistant | 0.7 | 0.9 | 1024 | 0.0–0.3 | 0.0–0.3 |
| Summarization | 0.3–0.5 | 0.9 | 800 | 0.3–0.5 | 0.0 |
| Creative writing | 1.0–1.2 | 0.95 | 2000 | 0.3 | 0.3 |
| Brainstorming | 1.2–1.5 | 0.95–1.0 | 1500 | 0.5 | 0.5 |

These are starting points. Always evaluate on your own data — optimal values depend on model, prompt, and domain.

---

## 18. Production Tuning Playbook

A reliable process for finding good parameters:

1. **Start conservative** — low temperature (0.0–0.3), top_p 0.9, modest max_tokens
2. **Measure task success** — accuracy, user ratings, automated evals on 50–200 examples
3. **Increase diversity only if needed** — if outputs feel too rigid, raise temperature in steps of 0.1
4. **Add penalties for repetition** — if loops appear, try frequency_penalty 0.3 first
5. **Set stop sequences** — for structured outputs, define explicit stop tokens
6. **Tune one parameter at a time** — changing everything at once makes debugging impossible
7. **Monitor cost and latency** — log tokens per request after each change

**Entropy–reliability trade-off:** Higher temperature and top_p increase exploration, which helps creativity but raises hallucination risk on factual tasks. Keep decoding conservative for high-stakes domains (medical, legal, financial).

---

## 19. Common Pitfalls

**Tuning on a single demo prompt** — parameters that look good on one example often fail on edge cases. Use a diverse eval set.

**Maximum creativity settings for factual tasks** — high temperature does not improve accuracy; it increases variance.

**Ignoring max_tokens** — runaway generation wastes money and frustrates users.

**Over-penalizing repetition** — frequency_penalty above 1.0 often breaks grammar and forces odd word choices.

**Assuming seed guarantees reproducibility** — model updates, API changes, and hardware differences can break replay.

**Changing temperature and top_p simultaneously** — hard to attribute effects; tune one at a time.

**No stop sequences for structured output** — models may emit extra prose after a JSON block or code fence.

---

## 20. Summary

Text generation at inference is a **decoding problem**: the model supplies logits, and your application transforms them into the next token through a pipeline of penalties, temperature scaling, truncation, and sampling.

**Temperature** controls how sharply probability concentrates on the top token — low for precision, high for creativity. **Top-k** and **top-p** truncate the tail of the distribution to avoid incoherent low-probability picks; top-p adapts to model confidence per step. **Greedy** decoding is deterministic; **sampling** enables variety; **beam search** optimizes full-sequence likelihood for translation-style tasks.

**Max tokens** and **stop sequences** bound output length and format. **Frequency penalty** discourages repeated tokens proportionally; **presence penalty** discourages any reuse. **Seed** supports reproducibility for debugging when the full stack is stable.

Tune for correctness first, then style. Log every parameter with each request, evaluate on representative data, and change one knob at a time. These inference controls do not replace grounding or verification — they shape how the model expresses what it has learned.